In [8]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning_scripts.lightning_classifier_matched_speech_in_noise import LitWordAudioSetModel
from lightning_scripts.jsinV3DataLoader_precombined_batched import jsinV3_precombined_all_signals, MatchedSpeechInNoiseDatasetBatched
from lightning_scripts.jsinV3DataLoader_precombined_batched import CleanSpeechInNoiseValDatasetBatched


sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path
import pickle
import h5py
import pandas as pd

In [5]:
## init config. Will be yaml eventually, but start as dict 
config_path = Path("model_configs/supervised_models/word_resnet18_MatchedDataset_LARS.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

TASK = "word"
# LAYER = "avgpool" ## ckpt at this layerneeds to exist 

# config['data'] = {}
# config['data']['root'] = "/mnt/ceph/users/jfeather/data/training_datasets_audio/JSIN_all_v3/subsets/"
config['num_workers'] = 4
config['hparas']['batch_size'] = 32
config['data']['eval_max'] = 3
# config['hparas']['optimizer'] = args.optimizer
# config['hparas']['lr'] = args.lr * args.gpus
# config['hparas']['epochs'] = 2
# don't load in classifier head if it exists 
# config['model']['arch_kwargs']['supervised'] =  False
# config['model']['arch_kwargs']['time_average'] = False

# if TASK == "word":
#     config['data']['task_label'] = 'signal/word_int'
#     config['model']['arch_kwargs']['num_classes'] = {"signal/word_int": 794} 
#     task_str = f"word_task"

# elif TASK == "speaker":
#     config['data']['task_label'] = 'signal/speaker_int'
#     config['model']['arch_kwargs']['n_classes'] =  433

config['hparas']['task_loss_params'] = {key:value for key,value in config['hparas']['task_loss_params'].items() if key in config['model']['arch_params']['num_classes'].keys()}


In [ ]:
ckpt_path = "model_checkpoints/word_resnet18_MatchedDataset_LARS/checkpoints/epoch=90-step=54600-best_word_task.ckpt"
# dummy init with checkpoint 
module = LitWordAudioSetModel.load_from_checkpoint(config=config, checkpoint_path=ckpt_path).eval()

# module.load_state_dict(classifier_ckpt['state_dict'])

module = module.cuda()
## update keys to remove _orig_mod from eatch key 

In [49]:


def collate_fn(batch):
    audio, targets = batch[0] # unbox wrapper added by dataloader 
    audio = audio.unsqueeze(1)
    # # combine labels: each target is dict for each key, stack the values 
    labels = {}
    for label_key in targets.keys():
        labels[label_key] = torch.from_numpy(targets[label_key])
    return audio, labels

In [11]:
# # run test 
test_dataset = CleanSpeechInNoiseValDatasetBatched(config['data']['speech_h5_path'],
                                            target_keys=config['data']['target_keys'],
                                            batch_size=100
)
# test_dataset.target_keys = ['signal/word_int']
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    num_workers=config['num_workers'],
    shuffle=False,
    collate_fn=collate_fn
)


In [12]:

audio, labels = next(iter(test_dataloader))

In [13]:
labels

{'signal/word_int': tensor([355, 606, 479,  29, 228, 729, 787,  11, 132, 764, 712, 331, 769, 769,
         254,  41, 496, 297, 741,  71,  98, 513, 345,  87, 184, 287, 471, 448,
          34, 581, 306, 397, 351, 200,  62, 769,  76, 580, 453,  36, 760,   2,
          34, 373, 625, 724, 680, 729, 752, 277, 635, 711, 101,  75, 785, 710,
         365, 769, 146, 209, 403, 145, 360, 269, 752, 453, 732,  31, 232, 729,
         213,  19, 747, 573, 118, 431, 632,  31,  49, 228, 453, 665, 594, 752,
         468, 456, 672, 519, 579,  18, 455, 397, 227,  57, 642, 493, 424, 278,
         201,  32])}

In [14]:
len(audio)

100

In [16]:

audio, labels = next(iter(test_dataloader))
word_labels = labels['signal/word_int']
module = module.cuda().eval()
with torch.no_grad():
    task_IXS = (word_labels != 0 ).nonzero(as_tuple=True)
    model_preds = module(audio.cuda())
    word_preds = model_preds['signal/word_int'].cpu().softmax(-1).cpu()[task_IXS]
    model_top_1 = word_preds.argmax(-1)
    word_labels = word_labels[task_IXS] 

## Cut audio to valid ixs for display
audio = audio[task_IXS]

In [17]:
raw_acc = (model_top_1 == word_labels).numpy().mean()
raw_acc

np.float64(0.93)

In [18]:
# model top5
top_5 = torch.isin(torch.topk(word_preds, k=5, dim=-1).indices, word_labels).any(-1).float().mean()
top_5

tensor(1.)

In [57]:
word_and_speaker_encodings = pickle.load(
    open("/mnt/home/igriffith/ceph/projects/cochdnn/robustness/audio_functions/word_and_speaker_encodings_jsinv3.pckl", "rb")
)
class_map = word_and_speaker_encodings["word_idx_to_word"]

In [58]:
### Geck examples where model predicted wrong label 




model_failure_IXS = torch.where((model_top_1 != word_labels))[0].numpy()
n_examples = min(len(model_failure_IXS), 20)

for _ in range(n_examples):

    failure_eg = int(model_failure_IXS[_])

    true_word = class_map[int(word_labels[failure_eg])]
    ## Get model top 1 and top 5 for that eg 
    model_pred = class_map[int(model_top_1[failure_eg])]

    # model top 5 transcripbed 
    model_eg_top5 = torch.topk(word_preds[failure_eg], k=5, dim=-1).indices
    model_eg_top5_words = [class_map[int(ix)] for ix in model_eg_top5] 

    print(f"True word: {true_word}")
    print(f"Model top 5 words: {', '.join(model_eg_top5_words)}")
    display(Audio(audio[failure_eg], rate=20_000, normalize=False))
    print("\n")


True word: eighty
Model top 5 words: about, eighty, without, eight, leading




True word: which
Model top 5 words: works, which, words, would, where




True word: known
Model top 5 words: group, known, number, development, currently




True word: dollars
Model top 5 words: million, dollars, dollar, billion, thousand




True word: eighty
Model top 5 words: nineteen, eighty, thirty, seventeen, ninety




True word: position
Model top 5 words: second, position, separate, mission, german




True word: ninety
Model top 5 words: because, become, ninety, nineteen, until


tensor([158, 244, 485, 421, 463])